## 1.0 Setup File

1.1 Import packages

In [61]:
import pandas as pd 
import requests # for making HTTP requests
from pathlib import Path
from urllib.parse import urlparse # for parsing URLs
import time

1.2 Read in .csv file

In [62]:
df = pd.read_csv('../Data/data.csv')

df.head()

,_id,coll_name,coll_place,description,lat,long,object_name,object_number,os_gridref,priref,repro_ref,strat_type,strat_unit,taxonomy,url
0,6696,unknown,Unknown,Fossilised remains of half of Silurian brachio...,NaN,NaN,Brachiopod,G.00400.001,unknown,654725,G.00400.001b.jpg; G.00400.001a.jpg,"Chronostratigraphy, Lithostratigraphy",Silurian Period; Aymestry Limestone Member,Kirkidium knightii,https://data.nhm.ac.uk/media/ludlow:G.00400.00...
1,7620,"Tarrant, P.; Tarrant, M.",Meadowley Hill,Fossilised remains of headshield of the Devoni...,52.533872,-2.487920,Fish,G.09365.002,SO6793,659204,G.09365.002a.jpg; G.09365.002b.jpg,Chronostratigraphy,Devonian Period; Ditton Series,Cephalaspis sp.,https://data.nhm.ac.uk/media/ludlow:G.09365.00...
2,6658,unknown,Unknown,Fossilised remains of the Silurian heliotid ta...,NaN,NaN,CORAL,G.00122,unknown,655457,G.00122a.jpg; G.00122b.jpg; G.00122c.jpg,"Chronostratigraphy, Lithostratigraphy",Silurian Period; Wenlock Series; Much Wenlock ...,Plasmopora sp.,https://data.nhm.ac.uk/media/ludlow:G.00122a.j...
3,6673,unknown,Mocktree,Fossilised remains of phragmacone of the Silur...,52.378964,-2.853479,Cephalopod,G.00183,SO4276,52039,G.00183.jpg,Chronostratigraphy,Silurian Period; Ludlow Series,Orthoceras ibex,https://data.nhm.ac.uk/media/ludlow:G.00183.jpg
4,6669,unknown,Buildwas,Fossilised complete shell of the Silurian orth...,52.632565,-2.533343,Brachiopod,G.00172,SJ6404,52030,G.00172.jpg,"Chronostratigraphy, Lithostratigraphy",Silurian Period; Wenlock Series; Wenlock Shale...,Dalejina hybrida,https://data.nhm.ac.uk/media/ludlow:G.00172.jpg


## 2.0 Clean Data File

2.1 Rename and Drop Columns

In [63]:
# Drop Unnecessary Columns
df.drop(columns=['coll_name', 'os_gridref','priref','repro_ref','coll_place','lat','long','object_number'], inplace=True)

In [64]:
# rename columns
df.rename(columns={'_id': 'fossil_id','strat_unit': 'period'}, inplace=True)

In [65]:
df.head()

,fossil_id,description,object_name,strat_type,period,taxonomy,url
0,6696,Fossilised remains of half of Silurian brachio...,Brachiopod,"Chronostratigraphy, Lithostratigraphy",Silurian Period; Aymestry Limestone Member,Kirkidium knightii,https://data.nhm.ac.uk/media/ludlow:G.00400.00...
1,7620,Fossilised remains of headshield of the Devoni...,Fish,Chronostratigraphy,Devonian Period; Ditton Series,Cephalaspis sp.,https://data.nhm.ac.uk/media/ludlow:G.09365.00...
2,6658,Fossilised remains of the Silurian heliotid ta...,CORAL,"Chronostratigraphy, Lithostratigraphy",Silurian Period; Wenlock Series; Much Wenlock ...,Plasmopora sp.,https://data.nhm.ac.uk/media/ludlow:G.00122a.j...
3,6673,Fossilised remains of phragmacone of the Silur...,Cephalopod,Chronostratigraphy,Silurian Period; Ludlow Series,Orthoceras ibex,https://data.nhm.ac.uk/media/ludlow:G.00183.jpg
4,6669,Fossilised complete shell of the Silurian orth...,Brachiopod,"Chronostratigraphy, Lithostratigraphy",Silurian Period; Wenlock Series; Wenlock Shale...,Dalejina hybrida,https://data.nhm.ac.uk/media/ludlow:G.00172.jpg


In [66]:
df.shape

(2617, 7)

2.2 Drop columns without period or url

In [67]:
# remove NA's from the period column
df.dropna(subset=['period'], inplace=True)
df.shape

(2367, 7)

In [68]:
# remove NA's from the url column
df.dropna(subset=['url'], inplace=True)
df.shape

(2367, 7)

In [69]:
# remove "unknown" from the period column
df = df[df['period'] != 'Unknown']
df.shape

(2358, 7)

2.3 Clean Period Column to Isolate Periods

In [70]:
# get list of all unique periods
periods = df['period'].unique()
print(periods)

['Silurian Period; Aymestry Limestone Member'
 'Devonian Period; Ditton Series'
 'Silurian Period; Wenlock Series; Much Wenlock Limestone Formation'
 'Silurian Period; Ludlow Series'
 'Silurian Period; Wenlock Series; Wenlock Shale Formation'
 'Ordovician Period' 'Devonian Period' 'Ordovician Period; Caradoc Series'
 'Silurian Period; Upper Bringewood Formation'
 'Devonian Period; Caithness Flagstone Group'
 'Silurian Period; Whitcliffe Formation; Ludlow Series'
 'Silurian Period; Ludlow Series; Middle Elton Formation'
 'Devonian Period; Caithness Flagstone Group; Achanarras Limestone Member'
 'Ordovician Period; Llandeilian Stage'
 'Ordovician; Ludlow Series; Upper Bringewood Formation' 'Silurian Period'
 'Silurian Period; Whitcliffe Formation'
 'Silurian Period; Elton Formation'
 'Silurian Period; Ludlow Series; Bringewood Formation'
 'Silurian Period; Lower Bringewood Formation'
 'Silurian Period; Ludlow Series; Whitcliffe Formation' 'Pridoli Series'
 'Permian Period' 'Silurian Peri

In [71]:
# drop everything after first semi-colon 
df['period'] = df['period'].str.split(';').str[0]

# remove leading and trailing whitespace
df['period'] = df['period'].str.strip()

# capitalizze using title case
df['period'] = df['period'].str.title()

# get list of all unique periods and number of fossils in each period
period_counts = df['period'].value_counts()
print(period_counts)

period
Silurian Period            982
Ordovician Period          337
Carboniferous Period       227
Devonian Period            224
Jurassic Period            121
Neogene Period             112
Precambrian Eon             74
Cretaceous Period           63
Silurian                    46
Triassic Period             41
Cambrian Period             28
Quaternary Period           22
Tertiary Period             15
Permian Period              14
Palaeogene Period           10
Paleogene Period             5
Cenozoic Era                 5
Eocene Epoch                 5
Pennsylvanian Subsystem      5
Ordovician                   2
Miocene                      2
Devonian                     2
Neogene                      2
Jurassic                     2
Pleistocene Epoch            2
Upper Carboniferous          2
Miocene Period               2
Pridoli Series               1
Carboniferous                1
Pliocene Period              1
Siliurian Period             1
Archaeon Eon                 1
N

In [72]:
# combine like periods
df['period'] = df['period'].replace({'Jurassic': 'Jurassic Period', 
                                    'Upper Carboniferous': 'Carboniferous Period',
                                    'Pennsylvanian Subsystem': 'Carboniferous Period',
                                    'Tertiary Period': 'Cenozoic Era',
                                    'Quaternary Period': 'Cenozoic Era',
                                    'Pleistocene Epoch': 'Cenozoic Era',
                                    'Pliocene Epoch': 'Cenozoic Era',
                                    'Silurian': 'Silurian Period',
                                    'Palaeogene Period': 'Cenozoic Era',
                                    'Neogene Period': 'Cenozoic Era',
                                    'Eocene Epoch': 'Cenozoic Era',
                                    'Paleogene Period': 'Cenozoic Era'})

# drop small periods
small_periods = period_counts[period_counts < 5].index
df = df[~df['period'].isin(small_periods)]

# get list of all unique periods and number of fossils in each period
period_counts = df['period'].value_counts()
print(period_counts)

period
Silurian Period         1028
Ordovician Period        337
Carboniferous Period     234
Devonian Period          224
Cenozoic Era             176
Jurassic Period          123
Precambrian Eon           74
Cretaceous Period         63
Triassic Period           41
Cambrian Period           28
Permian Period            14
Name: count, dtype: int64
